In [41]:
%pip install pennylane pennylane-lightning qiskit qiskit-aer matplotlib scipy pyqsp --break-system-packages

Note: you may need to restart the kernel to use updated packages.


In [42]:
import pennylane as qml
import numpy as np

def create_messy_VH(raw_matrix):
    """
    Constructs a mathematically perfect, but highly chaotic (1, 2, 0)-block-encoding.
    """
    # 1. Standardize H (Hermitian and Scaled)
    H_hermitian = (raw_matrix + raw_matrix.conj().T) / 2.0
    norm = np.linalg.norm(H_hermitian, ord=2)
    H = H_hermitian / (norm + 0.1) 
    
    # 2. Get PennyLane's "Lazy" Block Encoding
    dev = qml.device("default.qubit", wires=3)
    @qml.qnode(dev)
    def lazy_unitary():
        qml.BlockEncode(H, wires=[0, 1, 2])
        return qml.state()
    U_lazy = qml.matrix(lazy_unitary)()
    
    # 3. Build the Junk-Mixing Matrix (W)
    # We want a 4x4 matrix for the 2 ancilla qubits.
    # State |00> remains untouched (1.0). The other 3 states are mixed using a 3x3 DFT.
    W_anc = np.zeros((4, 4), dtype=complex)
    W_anc[0, 0] = 1.0 
    
    # A standard 3x3 unitary matrix to mix the junk states symmetrically
    w = np.exp(2j * np.pi / 3)
    W_junk_mixer = np.array([[1, 1, 1],
                             [1, w, w**2],
                             [1, w**2, w]]) / np.sqrt(3)
    
    W_anc[1:, 1:] = W_junk_mixer # Apply the mixer ONLY to the junk states
    
    # Expand W to apply to the whole 3-qubit system (W_anc ⊗ I_data)
    W_full = np.kron(W_anc, np.eye(2))
    
    # 4. Create the Messy Unitary!
    V_messy = W_full @ U_lazy
    
    return H, V_messy

In [43]:
# --- Run and Verify ---
messy_target = np.array([[2.0 + 1j, 3.0], 
                         [-1.0, 4.0 - 2j]])

H, V_H = create_messy_VH(messy_target)

np.set_printoptions(
    linewidth=150,     
    precision=3,       
    suppress=True,     
    formatter={'complexfloat': lambda x: f"{x.real:.3f}" if np.abs(x.imag) < 1e-8 else f"{x.real:.3f}+{x.imag:.3f}j"}
)

print(H, "\n\n")

print(V_H)


[[0.443 0.222]
 [0.222 0.886]] 


[[0.393 0.196 0.884 -0.159 0.000 0.000 0.000 0.000]
 [0.196 0.785 -0.159 0.565 0.000 0.000 0.000 0.000]
 [0.511 -0.092 -0.227 -0.113 0.577 0.000 0.577 0.000]
 [-0.092 0.326 -0.113 -0.453 0.000 0.577 0.000 0.577]
 [0.511 -0.092 -0.227 -0.113 -0.289+0.500j 0.000 -0.289+-0.500j 0.000]
 [-0.092 0.326 -0.113 -0.453 0.000 -0.289+0.500j 0.000 -0.289+-0.500j]
 [0.511 -0.092 -0.227 -0.113 -0.289+-0.500j 0.000 -0.289+0.500j 0.000]
 [-0.092 0.326 -0.113 -0.453 0.000 -0.289+-0.500j 0.000 -0.289+0.500j]]


In [44]:
import pennylane as qml
import numpy as np

def build_lcu_encoding(V_matrix):
    """
    Constructs the LCU circuit to calculate (I - H^2)/2 from a block-encoded H.
    """
    # Define our strict wire layout
    wire_order = ["C", "A1_0", "A1_1", "A2_0", "A2_1", "S"]
    dev_lcu = qml.device("default.qubit", wires=wire_order)

    @qml.qnode(dev_lcu)
    def lcu_circuit(V_mat):
        # 1. PREP
        qml.Hadamard(wires="C")
        
        # 2. SELECT (Double Block Encoding)
        qml.ctrl(qml.QubitUnitary, control="C")(V_mat, wires=["A1_0", "A1_1", "S"])
        qml.ctrl(qml.QubitUnitary, control="C")(V_mat, wires=["A2_0", "A2_1", "S"])
        
        # 3. Phase Correction
        qml.PauliZ(wires="C")
        
        # 4. PREP_dagger
        qml.Hadamard(wires="C")
        
        return qml.state()

    # THE FIX: Explicitly enforce the wire order when extracting the matrix!
    U_LCU = qml.matrix(lcu_circuit, wire_order=wire_order)(V_matrix)
    
    # Now [0:2, 0:2] accurately extracts the 'S' wire when the first 5 wires are |0>
    encoded_result = U_LCU[0:2, 0:2]
    
    return U_LCU, encoded_result

In [46]:
V_I_minus_H2_over_2, resulting_block = build_lcu_encoding(V_H)
H_encoded = V_H[0:2, 0:2]
math_target = (np.eye(2) - (H_encoded @ H_encoded)) / 2.0
print("H=")
print(H)
print("VH=")
print(V_H)
print("--- LCU Output ---")
print(resulting_block)
print("\n--- Math Target ---")
print(math_target)
print(f"\nMatch: {np.allclose(resulting_block, math_target)}")
print("\n--- Full LCU result ---")
print(V_I_minus_H2_over_2)

H=
[[0.443 0.222]
 [0.222 0.886]]
VH=
[[0.393 0.196 0.884 -0.159 0.000 0.000 0.000 0.000]
 [0.196 0.785 -0.159 0.565 0.000 0.000 0.000 0.000]
 [0.511 -0.092 -0.227 -0.113 0.577 0.000 0.577 0.000]
 [-0.092 0.326 -0.113 -0.453 0.000 0.577 0.000 0.577]
 [0.511 -0.092 -0.227 -0.113 -0.289+0.500j 0.000 -0.289+-0.500j 0.000]
 [-0.092 0.326 -0.113 -0.453 0.000 -0.289+0.500j 0.000 -0.289+-0.500j]
 [0.511 -0.092 -0.227 -0.113 -0.289+-0.500j 0.000 -0.289+0.500j 0.000]
 [-0.092 0.326 -0.113 -0.453 0.000 -0.289+-0.500j 0.000 -0.289+0.500j]]
--- LCU Output ---
[[0.404 -0.116]
 [-0.116 0.173]]

--- Math Target ---
[[0.404 -0.116]
 [-0.116 0.173]]

Match: True

--- Full LCU result ---
[[0.404 -0.116 -0.158 ... 0.000 0.000 0.000]
 [-0.116 0.173 -0.024 ... 0.000 0.000 0.000]
 [-0.091 -0.014 0.556 ... 0.000 0.000 0.000]
 ...
 [-0.039 0.058 -0.008 ... 0.583+0.144j 0.000 -0.167]
 [0.135 -0.039 -0.053 ... 0.000 0.583+0.144j 0.000]
 [-0.039 0.058 -0.008 ... -0.167 0.000 0.583+0.144j]]
